In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import tensorflow as tf
import datetime
!pip install transformers
!pip install datasets
import datasets
from datasets import load_dataset

/Users/sergioparigi/Desktop/TESI/Tesi Sergio Final/.venv3.9/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


/Users/sergioparigi/Desktop/TESI/Tesi Sergio Final/.venv3.9/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_dataset, DatasetDict


# Carica il dataset
dataset = load_dataset('csv', data_files='./stance_classification_dataset_final.csv', encoding = "utf-8", sep=',')

# Elimina le righe con NaN nella colonna 'Stance'
dataset = dataset.filter(lambda example: example['stance'] is not None)
dataset

DatasetDict({
    train: Dataset({
        features: ['folder', 'pulsar_file', 'id', 'search', 'content id', 'source', 'application', 'title', 'content', 'date', 'parent', 'language', 'url', 'parent source identifier', 'domain', 'credibility', 'credibility score', 'topics', 'image tags', 'tags', 'sentiment', 'sentiment class', 'sentiment by', 'main emotion', 'visibility', 'ave', 'duration', 'circulation', 'media reach', 'media impressions', 'social impressions', 'city', 'country', 'region', 'latitude', 'longitude', 'user city', 'user country', 'user latitude', 'user longitude', 'no. of followers', 'no. of friends', 'gender', 'bio', 'user views', 'Job titles', 'Companies', 'Industries', 'links url', 'no. of comments', 'no. of likes', 'no. of shares', 'no. of views', 'Linkedin reactions', 'Engagements/social shares', 'user name', 'user screen name', 'user source id', 'eRep', 'CSR', 'trust', 'CX', 'intensity', 'climate change', 'environmental impact', 'biodiversity', 'human rights', 'labo

In [3]:
dataset = dataset.filter(lambda example: example['stance'] != 4)
dataset

DatasetDict({
    train: Dataset({
        features: ['folder', 'pulsar_file', 'id', 'search', 'content id', 'source', 'application', 'title', 'content', 'date', 'parent', 'language', 'url', 'parent source identifier', 'domain', 'credibility', 'credibility score', 'topics', 'image tags', 'tags', 'sentiment', 'sentiment class', 'sentiment by', 'main emotion', 'visibility', 'ave', 'duration', 'circulation', 'media reach', 'media impressions', 'social impressions', 'city', 'country', 'region', 'latitude', 'longitude', 'user city', 'user country', 'user latitude', 'user longitude', 'no. of followers', 'no. of friends', 'gender', 'bio', 'user views', 'Job titles', 'Companies', 'Industries', 'links url', 'no. of comments', 'no. of likes', 'no. of shares', 'no. of views', 'Linkedin reactions', 'Engagements/social shares', 'user name', 'user screen name', 'user source id', 'eRep', 'CSR', 'trust', 'CX', 'intensity', 'climate change', 'environmental impact', 'biodiversity', 'human rights', 'labo

In [4]:
def transform_stance(example):
    example['stance'] = example['stance'] - 1
    return example

dataset = dataset.map(transform_stance)

In [5]:
dataset = dataset.cast_column('stance', datasets.ClassLabel(num_classes=5, names=[0,1,2,3,4]))

# Divide in training e test+validation, stratificando su 'Stance'
train_testvalid = dataset['train'].train_test_split(test_size=0.2, stratify_by_column='stance')

# Divide test+validation in test e validation, stratificando su 'Stance'
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, stratify_by_column='stance')

# Crea un nuovo DatasetDict con i tre set
dataset = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'valid': test_valid['train']
})

In [6]:
# Estrae il dataset dal dizionario
df_train = dataset['train']
df_valid = dataset['valid']
df_test = dataset['test']

In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


def tokenize_function(examples):
    texts = examples["content"]
    examples['labels'] = [label for label in examples['stance']]
    #examples['Stance'] = [int(label) if label is not None else -1 for label in examples['Stance']]
    texts = [str(text) for text in texts]
    return tokenizer(texts, padding="max_length", truncation=True)

In [8]:
df_train.column_names

['folder',
 'pulsar_file',
 'id',
 'search',
 'content id',
 'source',
 'application',
 'title',
 'content',
 'date',
 'parent',
 'language',
 'url',
 'parent source identifier',
 'domain',
 'credibility',
 'credibility score',
 'topics',
 'image tags',
 'tags',
 'sentiment',
 'sentiment class',
 'sentiment by',
 'main emotion',
 'visibility',
 'ave',
 'duration',
 'circulation',
 'media reach',
 'media impressions',
 'social impressions',
 'city',
 'country',
 'region',
 'latitude',
 'longitude',
 'user city',
 'user country',
 'user latitude',
 'user longitude',
 'no. of followers',
 'no. of friends',
 'gender',
 'bio',
 'user views',
 'Job titles',
 'Companies',
 'Industries',
 'links url',
 'no. of comments',
 'no. of likes',
 'no. of shares',
 'no. of views',
 'Linkedin reactions',
 'Engagements/social shares',
 'user name',
 'user screen name',
 'user source id',
 'eRep',
 'CSR',
 'trust',
 'CX',
 'intensity',
 'climate change',
 'environmental impact',
 'biodiversity',
 'human r

In [9]:
column_to_be_removed = ['folder',
 'pulsar_file',
 'id',
 'search',
 'content id',
 'source',
 'application',
 'title',
 #'content',
 'date',
 'parent',
 'language',
 'url',
 'parent source identifier',
 'domain',
 'credibility',
 'credibility score',
 'topics',
 'image tags',
 'tags',
 'sentiment',
 'sentiment class',
 'sentiment by',
 'main emotion',
 'visibility',
 'ave',
 'duration',
 'circulation',
 'media reach',
 'media impressions',
 'social impressions',
 'city',
 'country',
 'region',
 'latitude',
 'longitude',
 'user city',
 'user country',
 'user latitude',
 'user longitude',
 'no. of followers',
 'no. of friends',
 'gender',
 'bio',
 'user views',
 'Job titles',
 'Companies',
 'Industries',
 'links url',
 'no. of comments',
 'no. of likes',
 'no. of shares',
 'no. of views',
 'Linkedin reactions',
 'Engagements/social shares',
 'user name',
 'user screen name',
 'user source id',
 'eRep',
 'CSR',
 'trust',
 'CX',
 'intensity',
 'climate change',
 'environmental impact',
 'biodiversity',
 'human rights',
 'labor practices',
 'community and society',
 'workforce protection',
 'local content',
 'post subtype',
 'post type',
 'transcribed_text',
 'researcher',
 'no. of reposts',
 #'stance',
 'stance_name']

tokenized_datasets = df_train.map(
    tokenize_function,
    batched=True,
    remove_columns = column_to_be_removed
)

tokenized_datasets_valid = df_valid.map(
    tokenize_function,
    batched=True,
    remove_columns = column_to_be_removed
)

Map: 100%|██████████| 256/256 [00:00<00:00, 287.30 examples/s]


In [19]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="bert-base-uncased",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [21]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

In [22]:
from transformers import Trainer
import torch

# Auto-detect device
if torch.backends.mps.is_available():
    device = torch.device("mps")
    torch.mps.empty_cache()
elif torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.empty_cache()
else:
    device = torch.device("cpu")

trainer = Trainer(
    model=model,  # Trainer will auto-move model to device
    args=training_args,
    train_dataset=tokenized_datasets,
    eval_dataset=tokenized_datasets_valid,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print(f"Using device: {trainer.args.device}")

/var/folders/qq/95qwkbxs3zb9bw0w7n7tysmr0000gp/T/ipykernel_17880/1844951873.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Using device: mps


In [ ]:
trainer.train()
trainer.save_model("./BERT")

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.


/Users/sergioparigi/Desktop/TESI/Tesi Sergio Final/.venv3.9/lib/python3.9/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [ ]:
path = './BERT'
model = AutoModelForSequenceClassification.from_pretrained(path)
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
new_df = df_test.select_columns(['content', 'Stance']).to_pandas()
import torch

predictions = []
test = []
softmax_list = []
for i, row in new_df.iterrows():
  text = row["content"]
  test.append(row["Stance"])
  inputs = tokenizer(text, truncation=True, padding="longest", return_tensors="pt").to(device)
  with torch.no_grad():
    outputs = model(**inputs).logits
  paraphrased_text = torch.softmax(outputs, dim=1).tolist()[0]
  softmax_list.append(paraphrased_text)


In [ ]:
for elem in softmax_list:
  predictions.append(elem.index(max(elem)) + 1)

In [ ]:
from sklearn.metrics import classification_report
target_names = ['0', '1', '2', '3', '4']
print(classification_report(test, predictions, target_names = target_names))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        29
           1       0.26      0.61      0.37        18
           2       0.08      0.02      0.04        43
           3       0.03      0.50      0.05         2
           4       0.00      0.00      0.00         1

    accuracy                           0.14        93
   macro avg       0.07      0.23      0.09        93
weighted avg       0.09      0.14      0.09        93



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
